In [ ]:
# Build, validate, and save the reusable contract transformation pipeline and analytical dataset.
import sys, json; sys.path.append("..")
import pandas as pd, numpy as np, joblib
from src.contract_transform import ContractTransform, parse_date, FIT_CUTOFF
D = "../data/"
caps = json.load(open("../reports/caps.json")); woe_spec = json.load(open("../reports/woe_spec.json")); factor_spec = json.load(open("../reports/factor_spec.json"))

app = pd.read_csv(D + "applications.csv", low_memory=False).drop_duplicates("id", keep="first")
bur = pd.read_csv(D + "credit_bureau.csv", low_memory=False).drop(columns=["bureau_recovery_flag"])
ref = pd.read_parquet(D + "clean_v2.parquet").set_index("id")
raw = app.merge(bur, on="id", how="inner", validate="one_to_one")
raw = raw[raw.id.isin(ref.index)].reset_index(drop=True)
fit_mask = (parse_date(raw.issue_d) <= pd.Timestamp(FIT_CUTOFF)).to_numpy()
print("raw:", raw.shape, "| fit rows:", int(fit_mask.sum()))

ct = ContractTransform.build(raw[fit_mask], caps, woe_spec, factor_spec)
A = ct.transform(raw); A["defaulted"] = raw["defaulted"].to_numpy()
print("analytical:", A.shape)

cmp = A.set_index("id"); r = ref.loc[cmp.index]
for c in ["term", "annual_inc", "loan_amnt", "revol_util", "bc_util", "fico_range_low", "fico_range_high", "dti"]:
    print(f"{c:16} max |diff| vs notebooks:", float((cmp[c].astype(float) - r[c].astype(float)).abs().max()))
f1 = pd.Timestamp("1900-01-01")
print("state equal:", bool((cmp.addr_state == r.addr_state).all()), "| dates equal:", bool((cmp.earliest_cr_line.fillna(f1) == r.earliest_cr_line.fillna(f1)).all()),
      "| cr_line_invalid:", int(cmp.cr_line_invalid.sum()), int(r.cr_line_invalid.sum()))

F = A[fit_mask]
print(F.job_group.value_counts().to_dict())
print("factor std:", F[[f["name"] for f in factor_spec["factors"]]].std().round(2).to_list(), "| PCA variance:", ct.pca["var_ratio"].round(3).tolist())
W = [c for c in A if c.startswith("woe_")]
print("missing WoE values:", int(A[W].isna().sum().sum()))
print((F.groupby("defaulted")[W].mean().diff().iloc[-1]).round(3).sort_values().to_string())

joblib.dump(ct, "../contract/transform.joblib"); A.to_parquet("../contract/analytical_dataset.parquet", index=False)
num = [c for c in A.select_dtypes("number").columns if c != "defaulted"]
back = joblib.load("../contract/transform.joblib").transform(raw.head(200))
print("reload max diff:", float(np.nanmax(np.abs(back[num].astype(float).to_numpy() - A.loc[:199, num].astype(float).to_numpy()))))

In [ ]:
# Create a reference dataset containing 50 sample loan records to test and verify the transformation pipeline.
from src.contract_transform import RAW_INPUT, to_plain
rng = np.random.default_rng(0)
conds = {
 "odd date format": ~raw.issue_d.astype(str).str.fullmatch(r"[A-Za-z]{3}-\d{4}"),
 "income in thousands": (raw.annual_inc > 0) & (raw.annual_inc < 1000),
 "loan amount x10": raw.loan_amnt.to_numpy() != A.loan_amnt.to_numpy(),
 "fico swapped": raw.fico_range_low > raw.fico_range_high,
 "negative utilization": (raw.revol_util < 0) | (raw.bc_util < 0),
 "state spelling": raw.addr_state.to_numpy() != A.addr_state.astype(str).to_numpy(),
 "credit line after issue": A.cr_line_invalid == 1,
 "missing bc_util": raw.bc_util.isna(),
 "joint application": raw.application_type == "Joint App"}
idx = []
for name, m in conds.items():
    pool = np.flatnonzero(np.asarray(m, dtype=bool))
    pick = rng.choice(pool, min(3, len(pool)), replace=False) if len(pool) else []
    print(f"{name:26} rows in data: {len(pool):5} | picked: {len(pick)}"); idx += [int(i) for i in pick]
idx = list(dict.fromkeys(idx))
idx += [int(i) for i in rng.choice(np.setdiff1d(np.arange(len(raw)), idx), 50 - len(idx), replace=False)]
ref_out = pd.concat([raw.iloc[idx][RAW_INPUT].reset_index(drop=True),
                     to_plain(A.iloc[idx].drop(columns="defaulted")).add_prefix("exp_").reset_index(drop=True)], axis=1)
ref_out.to_csv("../contract/reference_transforms.csv", index=False); print(ref_out.shape)

In [ ]:
# Build and save the data contract documenting input schemas, validation rules, preprocessing, and excluded features.
import json, os
from src.contract_transform import CONTRACT_VERSION, FIT_CUTOFF, RAW_INPUT
from src.treat import HEAVY
mf = caps["missing"]
T = [
 ("id", "str", None, None, "unique loan id", "reject", "high", "unique after 600 repeated ids removed"),
 ("issue_d", "date", "2013-01-01", "2016-12-31", "parsed from Mon-YYYY, YYYY-MM-DD or MM/YYYY", "reject", "high", "0 parse failures; 1,200 odd formats repaired"),
 ("loan_amnt", "float", 1000, 40000, "divided by 10 when 10x the amount implied by installment, rate and term", "reject", "high", "300 inflated amounts repaired; payment formula matches the rest"),
 ("term", "int", 36, 60, "trimmed; digits extracted ('36 months' -> 36)", "reject", "high", "only 36 and 60"),
 ("int_rate", "float", 5, 31, "none", "reject", "high", "consistent with installment; platform output, not independent"),
 ("installment", "float", 1, 1700, "none", "reject", "high", "matches payment formula; 12 inconsistent loans dropped"),
 ("grade", "category", None, None, "trimmed", "reject", "high", "accurate; platform output, excluded from the main model"),
 ("sub_grade", "category", None, None, "trimmed", "reject", "high", "accurate; platform output, excluded from the main model"),
 ("purpose", "category", None, None, "trimmed; 9 rare purposes merged into other (<300 loans)", "reject", "medium", "borrower-selected; no inconsistencies found"),
 ("application_type", "category", None, None, "trimmed", "reject", "medium", "only Individual before 2015; Joint App appears in 2015 (16) and 2016 (96)"),
 ("emp_length", "category", None, None, "mapped to <1y, 1-9y, 10+y", "own category MISSING", "low", "self-reported; 4.8% blank; weak signal (IV 0.010)"),
 ("home_ownership", "category", None, None, "trimmed", "reject", "medium", "self-reported; 50% say MORTGAGE, 61% have a mortgage account"),
 ("annual_inc", "float", 0, 10000000, "values below 1,000 treated as thousands (x1000)", "reject", "medium", "self-reported; about half are round figures; 899 unit errors fixed"),
 ("verification_status", "category", None, None, "trimmed", "reject", "medium", "records the platform's choice of whom to verify, not a borrower trait"),
 ("dti", "float", 0, 100, "no cap (loans above the old p99.5 default at 34.5%)", "reject", "medium", "computed from stated income; shifted up in 2015; 3 values above 100 seen (1 in 2015, 2 in 2016)"),
 ("addr_state", "category", None, None, "trimmed, uppercased, full names mapped to codes; 31 small states merged", "reject", "high", "accurate but sensitive; not a model input, monitoring only"),
 ("earliest_cr_line", "date", "1940-01-01", "2016-12-31", "three date formats; set missing + flag cr_line_invalid if after issue_d", f"median history {caps['features']['credit_history_months_fill']:.0f} months", "medium", "300 impossible dates removed; shifted dates that stay valid cannot be detected"),
 ("fico_range_low", "float", 300, 850, "swapped with high when low > high", "reject", "high", "150 swaps repaired; high = low + 4 in all other rows"),
 ("fico_range_high", "float", 300, 850, "swapped with low when low > high", "reject", "high", "see fico_range_low"),
 ("inq_last_6mths", "int", 0, 50, "none", "reject", "high", "bureau count; no contradictions"),
 ("delinq_2yrs", "int", 0, 50, "none", "reject", "high", "bureau count; no contradictions"),
 ("mths_since_last_delinq", "float", 0, 200, "none", "blank means never delinquent; flag never_delinquent", "medium", "46% blank, structural; 0.27% contradict delinq_2yrs"),
 ("pub_rec", "int", 0, 50, "none", "reject", "high", "bureau count"),
 ("pub_rec_bankruptcies", "int", 0, 20, "none", "reject", "high", "bureau count; never above pub_rec"),
 ("open_acc", "int", 0, 100, "none", "reject", "high", "never above total_acc"),
 ("total_acc", "int", 0, 200, "none", "reject", "high", "bureau count"),
 ("revol_bal", "float", 0, 5000000, "none", "reject", "high", "40 real zero balances; heavy tail is genuine"),
 ("revol_util", "float", 0, 500, "absolute value", f"median {mf['revol_util']['fill']:.2f} (fit set)", "high", "sign flips repaired; matches balance/limit recomputed"),
 ("bc_util", "float", 0, 500, "absolute value", f"median {mf['bc_util']['fill']:.2f} (fit set) + flag bc_util_missing", "medium", "sign flips assumed fixed by analogy; no limit column to recompute"),
 ("tot_cur_bal", "float", 0, 10000000, "none", "reject", "high", "heavy tail is genuine (mortgage holders)"),
 ("avg_cur_bal", "float", 0, 1000000, "none", "reject", "medium", "approximately tot_cur_bal / open_acc (derived)"),
 ("total_rev_hi_lim", "float", 0, 5000000, "none", "reject", "high", "bureau limit"),
 ("tot_hi_cred_lim", "float", 0, 10000000, "none", "reject", "high", "bureau limit"),
 ("mort_acc", "int", 0, 50, "none", "reject", "high", "bureau count"),
 ("acc_open_past_24mths", "int", 0, 100, "none", "reject", "high", "bureau count"),
 ("num_tl_op_past_12m", "int", 0, 50, "none", "reject", "high", "bureau count"),
 ("emp_title", "str", None, None, "lowercased, punctuation removed, mapped to 10 job groups by keyword; groups under 300 loans merged into other", "own group unknown", "low", "free text, 9,093 distinct values"),
]
ALLOWED = {c: sorted(raw[c].dropna().astype(str).str.strip().unique()) for c in ["grade", "sub_grade", "purpose", "application_type", "emp_length", "home_ownership", "verification_status"]}
fields = []
for name, typ, lo, hi, clean, miss, rel, why in T:
    f = {"name": name, "type": typ, "min": lo, "max": hi, "cleaning": clean, "missing_strategy": miss, "reliability": rel, "reliability_reason": why}
    if name in ALLOWED: f["allowed_values"] = ALLOWED[name]
    cap = (caps.get(name) or {}).get("cap_p99.5")
    if cap: f["cleaning"] += f"; capped at p99.5 = {cap:,.2f}" + (", then log1p" if name in HEAVY else "")
    fields.append(f)
assert set(f["name"] for f in fields) == set(RAW_INPUT), set(RAW_INPUT) ^ set(f["name"] for f in fields)
SERV = ["total_pymnt", "total_rec_prncp", "total_rec_int", "recoveries", "last_pymnt_d", "last_pymnt_amnt", "last_credit_pull_d", "last_fico_range_high", "last_fico_range_low", "debt_settlement_flag"]
p = ct.pca
spec = {"contract_version": CONTRACT_VERSION, "fit_cutoff": FIT_CUTOFF, "fields": fields,
 "excluded_columns": {"outcome_derived": ["bureau_recovery_flag"], "post_issue_servicing": SERV,
                      "reason": "recorded after the loan was issued; bureau_recovery_flag defaults 100% when 1 and is always 0 in the holdout, AUC 0.874 alone"},
 "ignored_input_columns": ["bureau_recovery_flag"],
 "not_model_inputs": {"zip_code": "sparse (median 10 loans per zip3) and a proxy risk", "addr_state": "IV 0.013; proxy risk; used only to monitor outcomes by region",
                      "title": "free text, redundant with purpose", "application_type": "single category in training; out-of-domain when Joint App"},
 "input_rules": ["installment must match loan_amnt, int_rate and term within $1 (after the x10 repair) or the record fails validation",
                 "term is 36 or 60; Joint App is scored but flagged out_of_domain", "dti outside 0-100 is capped and flagged"],
 "factors": factor_spec["factors"],
 "pca": {"n_components": len(p["components"]), "variance_ratio": [round(float(v), 4) for v in p["var_ratio"]], "input_columns": p["columns"]},
 "woe": woe_spec, "caps": caps}
json.dump(spec, open("../contract/data_spec.json", "w"), indent=1, default=str)
print(len(fields), "fields |", len(spec["factors"]), "factors |", len(woe_spec["scorecard_vars"]), "scorecard variables")
print({f: os.path.getsize("../contract/" + f) // 1024 for f in os.listdir("../contract")}, "KB")